# DSE 200 FINAL PROJECT
## Fall 2025
### Due Date: November 21th, 2025

This project is culmination of all you’ve learned in this course! You should expect to spend <b>24-32 total hours</b> on the project. Be sure to read all of the items below before starting.

There are a number of steps outlined below, but is critical that you do not view this as an entirely linear process.  Remember that the science component in data science is the creation of a hypothesis based on exploration and testing of that hypothesis through analysis.  You may need to go through many of these steps multiple times before you arrive at meaningful hypothesis or conclusions.

## Step 1: Find a dataset or datasets

Based on your interest, identify a dataset which you will want to examine.  You will find a starting point for where you can find open datasets at the end of this notebook, but feel free to use other datasets you have access to and can publicly share results about.


This step may take some time, as you’ll likely look at a number of datasets before you find one (or more) which holds promising data for the kinds of questions you want to ask. You are expected to use at least two interconnected datasets, e.g., two tables in one database or a combination of datasets which you can merge in some meaningful way.


In [1]:
#EXPLAIN AND INGEST YOUR DATASET IN THIS SECTION

Let's import what we'll need

In [ ]:
!pip install backtesting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import re
import seaborn as sns

Donwload what we need from kaggle

In [ ]:

# Download latest version
path = kagglehub.dataset_download("mattiuzc/stock-exchange-data")

path2 = kagglehub.dataset_download("saketk511/world-important-events-ancient-to-modern")

print("Path to dataset files:", path)

#with open(path)

In [ ]:
os.listdir(path)
os.listdir(path2)

In [ ]:
with open(os.path.join(path2, "World Important Dates.csv"), "r", encoding="utf-8") as f:
    for i in range(20):
        print(f.readline())


Load the Data info dataframes

In [ ]:
df_data = pd.read_csv(os.path.join(path, "indexData.csv"))
df_info = pd.read_csv(os.path.join(path, "indexInfo.csv"))
df_processed = pd.read_csv(os.path.join(path, "indexProcessed.csv"))
# df_events = pd.read_csv(os.path.join(path2, "World Important Dates.csv"))

df_events = pd.read_csv(
    os.path.join(path2, "World Important Dates.csv"),
    engine="python",
    quotechar='"',
    skipinitialspace=True
)
# df_events = pd.read_csv(
#     os.path.join(path2, "World Important Dates.csv"),
#     engine="python",
#     quotechar='"',
#     skipinitialspace=True,
#     on_bad_lines="skip"    # or "warn"
# )


## Step 2: Explore the datasets

In this step, you should explore what is present in the data and how the data is organized. You’ll need to determine what common features allow you to merge the datasets.  

You are expected to answer the following questions using the _pandas_ library and markdown cells to describe your actions:

* Are there quality issues in the dataset (noisy, missing data, etc.)?
* What will you need to do to clean and/or transform the raw data for analysis?

You are also expected to use the _matplotlib_ library to visually explore the datasets and explain your findings, specifically,

* How are the data distributed?
* What are some common trends?
* What are the relationships between variables in your datasets?

In [ ]:
#PERFORM AND EXPLAIN YOUR EXPLORATORY ANALYSIS IN THIS SECTION

### Data overview
indexData.csv: daily price + volume data for major global indices

indexInfo.csv: metadata: region, exchange, currency, index name

indexProcessed.csv: cleaned version of the price data with USD-normalized closes (CloseUSD)

In [ ]:

df_data.head()

In [ ]:
df_info.head()

In [ ]:
df_processed.head()

In [ ]:
df_events.head()

### Explaination of data analysis
- Looking for data shape, content, & quality issues

### Check their shapes

In [ ]:
df_data.shape, df_info.shape, df_processed.shape, df_events.shape

From the shape we can see that df_data and df_processed the majority of info and processed likely has fewer rows since it already been cleaned.

In [ ]:
df_data.sample()

### Check their column content

In [ ]:
df_data.info(), df_info.info(), df_processed.info(), df_events.info()

From the shape we can see the majority of columns are numeric except for df_info.

Next we'll take a look at how many missing values we have to deal with.

In [ ]:
df_data.isnull().sum(), df_info.isnull().sum(), df_processed.isnull().sum(), df_events.isnull().sum()

Plotting the missingingness of the data we see no column is greater than 2% missing data.

In [ ]:
import matplotlib.pyplot as plt
(df_data.isnull().mean()*100).plot(kind="bar", figsize=(10,4))
plt.title("Percentage of Missing Values by Column")

Check the index

In [ ]:
df_data.Index.unique()

From the review we can see that while df_data has the only missing values which also are only numeric columns, the df_processed has been cleaned of all the missing data issues.
Next we'll check the size of the dataframe before the merge and then after to make sure we don't have any loss.

In [ ]:
df_data.shape[0]

In [ ]:
df_events.shape[0]

Join the df_data with df_info

In [ ]:
df_data_combined = df_data.merge(df_info, how= 'left', on='Index')
df_data_combined.shape[0]

# Clean up the Events Dataframe to Join with the data_combined Dataframe

In [ ]:
df_events.columns

In [ ]:
# To properly join events with data_combined dataframe we'll need to create a data column to join on
# Step A: evaluate the Year, Month, & Date to see if anything needs cleaned before joining

In [ ]:
for col in ['Year', 'Month', 'Date']:
    print(f"\nUnique {col} values:")
    print(df_events[col].unique())
    print("\nCounts:")
    print(df_events[col].value_counts(dropna=False))


We can see that Day values have several non-day, missing, and unknown values that need to be cleaned before merging with the rest of the columns.
We also see that there are months that are showing as NANs as well, without the exact month dropping instead of assigning Jan or July would probably be safest.

In [ ]:
# Normalize the text
df_events['Year_str'] = df_events['Year'].astype(str).str.upper().str.strip()
#Remove the BC
def parse_year(val):
    # BC case
    if 'BC' in val:
        num = ''.join([c for c in val if c.isdigit()])
        return -int(num) if num else None

    # AD or unknown case
    digits = ''.join([c for c in val if c.isdigit()])
    return int(digits) if len(digits) == 4 else None

# for all rows for years in string form apply parse_year function
df_events['Year_clean'] = df_events['Year_str'].apply(parse_year)

#Keep only modern events
df_events = df_events[df_events['Year_clean'] >= 1900]

In [ ]:
# Function to clean the day (date) column
def clean_day(val):
    if not isinstance(val, str):
        return np.nan

    val = val.strip()

    if val.lower() in ['unknown', '', 'nan']:
        return np.nan

    if re.fullmatch(r'\d{4}', val):   # incorrectly placed year
        return np.nan

    match = re.search(r'\b([1-9]|[12][0-9]|3[01])\b', val)
    if match:
        return int(match.group(0))

    if re.fullmatch(r'[A-Za-z]+', val):  # month names in wrong column
        return np.nan

    return np.nan
df_events['Day_clean'] = df_events['Date'].apply(clean_day)
df_events['Day_clean'] = df_events['Day_clean'].fillna(1)

month_map = {
    'January':1, 'February':2, 'March':3, 'April':4,
    'May':5, 'June':6, 'July':7, 'August':8,
    'September':9, 'October':10, 'November':11, 'December':12
}

df_events['Month_num'] = df_events['Month'].map(month_map)

df_events = df_events.dropna(subset=['Month_num'])

df_events['Event_Date'] = pd.to_datetime(
    dict(
        year=df_events['Year_clean'],
        month=df_events['Month_num'],
        day=df_events['Day_clean']
    ), errors='coerce'
)

df_events['Event_Date_str'] = df_events['Event_Date'].dt.strftime('%Y-%m-%d')


Here we will also filter for US Index

Since there are multiple events per date so a simple join on date wouldn't work well so we will take the following steps:
1. Create an event map of about 10 event groups
2. Map the groups to df_events['Type of Event']
3. Combine the events on the same date into one row
4. Encode the grouped events into indicator features for clustering and modelling

In [ ]:
event_type_map = {

    # Political
    'Political': 'political', 'Political Action': 'political',
    'Political Agreement': 'political', 'Political Change': 'political',
    'Political Corruption': 'political', 'Political Coup': 'political',
    'Political Crisis': 'political', 'Political Declaration': 'political',
    'Political Development': 'political', 'Political Movement': 'political',
    'Political Party': 'political', 'Political Party Formation': 'political',
    'Political Reform': 'political', 'Political Repression': 'political',
    'Political Revolution': 'political', 'Political Scandal': 'political',
    'Political Transition': 'political', 'Political Unification': 'political',
    'Political Union': 'political', 'Political Uprising': 'political',
    'Political/Economic Integration': 'political',
    'Political/Legal': 'political', 'Political/Secession': 'political',
    'Political/Territorial Change': 'political',
    'Legislation': 'political', 'Legislative': 'political',
    'General Election': 'political', 'General Elections': 'political',
    'Referendum': 'political', 'Election': 'political',
    'Declaration of Independence': 'political',
    'Independence Declaration': 'political',

    # Military
    'Military': 'military_conflict', 'Military Administration': 'military_conflict',
    'Military Aggression': 'military_conflict', 'Military Battle': 'military_conflict',
    'Military Campaign': 'military_conflict', 'Military Conflict': 'military_conflict',
    'Military Coup': 'military_conflict', 'Military Engagement': 'military_conflict',
    'Military Intervention': 'military_conflict', 'Military Invasion': 'military_conflict',
    'Military Occupation': 'military_conflict', 'Military Offensive': 'military_conflict',
    'Military Operation': 'military_conflict', 'Military Rebellion': 'military_conflict',
    'Military Siege': 'military_conflict', 'Military Surrender': 'military_conflict',
    'Military/Atomic Warfare': 'military_conflict',
    'Military/Political': 'military_conflict',
    'Military/Political Event': 'military_conflict',
    'Military/Political Incident': 'military_conflict',
    'War': 'military_conflict', 'War Declaration': 'military_conflict',
    'Regional Conflict': 'military_conflict', 'Civil War': 'military_conflict',
    'Rebellion': 'military_conflict', 'Revolution': 'military_conflict',
    'Coup': 'military_conflict', "Coup d'état": 'military_conflict',

    # Terrorism
    'Terrorism': 'terror', 'Domestic Terrorism': 'terror',

    # Disasters
    'Natural Disaster': 'disaster', 'Natural Event': 'disaster',
    'Industrial Disaster': 'disaster', 'Industrial Accident': 'disaster',
    'Nuclear Accident': 'disaster', 'Space Exploration Disaster': 'disaster',
    'Aviation': 'disaster', 'Accident': 'disaster',
    'Maritime Disaster': 'disaster',

    # Economic
    'Economic': 'economic', 'Economic Crisis': 'economic',
    'Economic Development': 'economic', 'Economic Integration': 'economic',
    'Economic Reform': 'economic', 'Economic Policy': 'economic',
    'Central Banking Institution': 'economic',
    'International Economic Agreement': 'economic',
    'International Economic Integration': 'economic',
    'International Trade Membership': 'economic',
    'Financial Technology': 'economic',

    # Diplomacy & Treaties
    'Diplomatic': 'diplomacy', 'Diplomatic Agreement': 'diplomacy',
    'Diplomatic Event': 'diplomacy', 'Diplomatic Incident': 'diplomacy',
    'Diplomatic Meeting': 'diplomacy', 'Diplomatic Statement': 'diplomacy',
    'International Agreement': 'diplomacy', 'International Cooperation': 'diplomacy',
    'International Organization': 'diplomacy', 'International Treaty': 'diplomacy',
    'Peace Agreement': 'diplomacy', 'Peace Process': 'diplomacy',
    'Treaty': 'diplomacy', 'Negotiation': 'diplomacy',

    # Social Movements / Unrest
    'Social Movement': 'social_movement', 'Social Unrest': 'social_movement',
    'Civil Disobedience': 'social_movement',
    'Civil Disobedience Movement': 'social_movement',
    'Civil Rights': 'social_movement', 'Civil Unrest': 'social_movement',
    'Mass Protest': 'social_movement', 'Protest Movement': 'social_movement',
    'Protests': 'social_movement', 'Democratic Uprising': 'social_movement',
    'Violent Protest': 'social_movement',

    # Crime / Violence
    'Criminal Incident': 'crime_violence', 'Massacre': 'crime_violence',
    'Gun Violence': 'crime_violence', 'Genocide': 'crime_violence',

    # Science & Technology
    'Scientific': 'science_tech', 'Technology': 'science_tech',
    'Technological Advancement': 'science_tech',
    'Space Exploration': 'science_tech', 'Space Agency': 'science_tech',
    'Engineering Achievement': 'science_tech', 'Medical': 'science_tech',

    # Infrastructure
    'Infrastructure': 'infrastructure', 'Urban Development': 'infrastructure',
    'Telecommunications': 'infrastructure', 'Rail': 'infrastructure',

    # Cultural
    'Cultural': 'cultural', 'Cultural Celebration': 'cultural',
    'Cultural Change': 'cultural', 'Cultural Infrastructure': 'cultural',
    'Commemoration': 'cultural', 'World Expo': 'cultural',
    "World's Fair": 'cultural',

    # Environmental
    'Environmental': 'environmental', 'Environmental/Economic Policy': 'environmental',
    'Environmental/Social': 'environmental',
    'Wildlife Conservation': 'environmental',

    # Health
    'Health': 'health', 'Public Health': 'health',

    # Geopolitics / Sovereignty
    'Annexation': 'sovereignty_geopolitical', 'Country Formation': 'sovereignty_geopolitical',
    'State Establishment': 'sovereignty_geopolitical', 'State Formation': 'sovereignty_geopolitical',
    'Union': 'sovereignty_geopolitical', 'Independence': 'sovereignty_geopolitical',
    'Partition': 'sovereignty_geopolitical', 'Sovereignty Transition': 'sovereignty_geopolitical',

    # Sports
    'Sport': 'sport', 'Sports': 'sport',
    'Sporting Event': 'sport', 'International Sports Event': 'sport',

    # Everything else
    'Administrative': 'other', 'Historical': 'other', 'Monument': 'other',
    'Media': 'other', 'Era Change': 'other'
}


In [ ]:
# Step 2: create a mapping function to map the groups to df_events
def map_event_type(event):
    if pd.isna(event):
        return 'other'
    return event_type_map.get(event, 'other')
df_events['Event_Type_Mapped'] = df_events['Type of Event'].apply(map_event_type)


In [ ]:
# verify mapping worked
df_events['Event_Type_Mapped'].value_counts()

In [ ]:
# Filter events for US events / globally relevant events
df_us = df_events[df_events['Country'] == 'United States']

global_relevant = [
    'military_conflict','terror','economic','political',
    'sovereignty_geopolitical','diplomacy','infrastructure', 'environmental'
]

df_global = df_events[df_events['Event_Type_Mapped'].isin(global_relevant)]
df_events_filtered = pd.concat([df_us, df_global]).drop_duplicates()



In [ ]:
# 3: Combine them into one row per date
df_events_grouped = (
    df_events_filtered
    .groupby('Event_Date')
    .agg({
        'Event_Type_Mapped': list,
        'Outcome': list,
        'Impact': list
    })
    .reset_index()
)


In [ ]:
df_events_grouped.head()

In [ ]:
#4: Create dummy features to encode before merging to df_combined

# 4a. Extract the event types
all_event_types = sorted(df_events['Event_Type_Mapped'].unique())
all_event_types



In [ ]:
# 4b.
for event_type in all_event_types:
    df_events_grouped[f"event_{event_type}"] = df_events_grouped['Event_Type_Mapped'].apply(
        lambda x: 1 if event_type in x else 0
    )
df_events_grouped.head()


Now that the dates are down to one row and encoded we'll join the event_df with the combined_df

In [ ]:
# Ensure both are datetime
df_data_combined['Date'] = pd.to_datetime(df_data_combined['Date'])
df_events_grouped['Event_Date'] = pd.to_datetime(df_events_grouped['Event_Date'])

In [ ]:
df_data_combined.shape[0]

In [ ]:
df_data_index = df_data_combined.merge(
    df_events_grouped.drop(columns=['Event_Type_Mapped', 'Outcome', 'Impact']),
    how='left',
    left_on='Date',
    right_on='Event_Date'
    )

In [ ]:
df_data_index.shape[0]

Check duplicates

In [ ]:
df_data_index.duplicated().sum()
# none exist

Next we'll check all the data types for feature correlation

In [ ]:
# numeric columns
print('All columns: ', df_data_index.columns,'\n')

print('Numeric columns: ', df_data_index.select_dtypes(include="number").columns)

Check each numeric variable's distribution for skew, outiers or dominant categories

In [ ]:
df_data_index.select_dtypes(include="number").describe()

In [ ]:
# Open high, low, close, adj close are fairly similar so will just plot 1
# create two histograms to evaluate the layout of the data
plt.hist(df_data_index['Close'], bins=40, alpha=0.6)
plt.title('Distribution of Close Prices')
plt.xlabel("Close")
plt.ylabel("Frequency")
plt.xticks(
    ticks=range(0, 70001,5000),
           rotation=45
)
plt.show()

In [ ]:
# Volume has unique values so will plot it
plt.hist(df_data_index["Volume"],bins=20,alpha=0.6)
plt.title("Distribution of Volume")
plt.xlabel("Volume")
plt.ylabel("Frequncy")
# plt.xticks(
#     ticks=range(0,8,2),
#     rotation=45
# )
plt.show()

# since we see a large right skew we'll use log to trasform the data

plt.hist(
    df_data_index['Volume'][df_data_index['Volume'] > 0].apply(np.log10),
    bins=40, alpha=0.6
)

plt.title("Distribution of Volume (log10)")
plt.xlabel("log10(Volume)")
plt.ylabel("Frequency")
plt.show()


Numeric vs. Categorical Analysis

In [ ]:
numeric_columns = df_data_index.select_dtypes(include=["number"]).columns
categorical_columns = df_data_index.select_dtypes(exclude=["number"]).columns

In [ ]:
categorical_columns

In [ ]:
df_data_index['Daily_Return'] = df_data_index.groupby('Index')['Close'].pct_change()

plt.figure(figsize=(12,6))
sns.boxplot(x='Index', y='Daily_Return', data=df_data_index)
plt.xticks(rotation=90)
plt.title("Daily Return Distribution by Index")
plt.show()
'''NOTE:
Indexes with a wide IQR (interquartile range) are riskier. Indexes with a tight IQR are steadier.
The typical daily return is close to zero, but the indices differ heavily in volatility. HSI, TWII, and 399001.SZ show larger day-to-day swings, while N100 and GDAXI are comparatively stable.
'''
df_data_index['log_volume'] = np.log10(df_data_index['Volume'])

plt.figure(figsize=(14,6))
sns.boxplot(x='Index', y='log_volume', data=df_data_index)
plt.xticks(rotation=45)
plt.title("Log10 Volume Distribution by Index")
plt.show()

'''
After converting volume to a log scale, GSPTSE has the widest box in the plot (around log10 values 7–10). That means its daily trading volume swings a lot —
from about 10 million shares on some days to over 10 billion on others. This shows big changes in how much the market trades, not big swings in the index’s price.
'''

Price variables all move together (corr ≈ 1.0), so they’re redundant. Volume barely correlates with price (≈ 0.14). To get more insight, we need to examine distribution patterns by index or create additional features like daily returns or volatility.

In [ ]:
df_data_index.head()

In [ ]:
# Adding in more features
# Volatility_5D = std of last 5 daily returns
# how do I do every five day rotation?
df_data_index['Volatility_5D'] = (df_data_index.groupby('Index')['Daily_Return'].rolling(window=5).std().reset_index(level=0, drop=True))

# Volatility_10D
df_data_index['Volatility_10D'] = (df_data_index.groupby('Index')['Daily_Return'].rolling(window=10).std().reset_index(level=0, drop=True))

# Rolling_Avg_Close_50D
df_data_index['Rolling_Avg_Close_50D'] = (df_data_index.groupby('Index')['Close'].rolling(window=50).mean().reset_index(level=0, drop=True))

# Rolling_Avg_Close_200D
df_data_index['Rolling_Avg_Close_200D'] = (df_data_index.groupby('Index')['Close'].rolling(window=200).mean().reset_index(level=0, drop=True))

# Volume_Change (pct change)
df_data_index['Volume_Change'] = (df_data_index.groupby('Index')['Volume'].pct_change())

# Range = High − Low
df_data_index['Range'] =  df_data_index['High'] - df_data_index['Low']

# Intraday_Volatility = (High − Low) / Open
df_data_index['Intraday_Volatility'] =  (df_data_index['High'] - df_data_index['Low']) / (df_data_index['Open'])

# RSI

# Momentum Indicator

# Prediction value = Profit



In [ ]:
# 1) Sort by date just to be safe
df_data_index = df_data_index.sort_values("Date")

# 2) Create forward returns
df_data_index['Event_Return_1D'] = df_data_index['Close'].pct_change().shift(-1)
df_data_index['Event_Return_5D'] = df_data_index['Close'].pct_change(5).shift(-5)
df_data_index['Event_Return_30D'] = df_data_index['Close'].pct_change(30).shift(-30)

# 3) Event-period volatility
returns = df_data_index['Close'].pct_change()

df_data_index['Event_Vol_5D']  = returns.shift(-1).rolling(5).std()
df_data_index['Event_Vol_10D'] = returns.shift(-1).rolling(10).std()


# check the new features
engineered_features = [
    'Daily_Return', 'Volatility_5D', 'Volatility_10D',
    'Rolling_Avg_Close_50D', 'Rolling_Avg_Close_200D',
    'Volume_Change', 'Range', 'Intraday_Volatility'
]
reaction_features = [
    'Event_Return_1D', 'Event_Return_5D', 'Event_Return_30D',
    'Event_Vol_5D', 'Event_Vol_10D'
]
other_features = ['Volume', 'Close'
]

# Combine into one flat list
all_features = engineered_features + reaction_features + other_features

corr_matrix = df_data_index[all_features].corr()

corr_matrix_rounded = corr_matrix.round(2)

plt.figure(figsize=(18, 14))  # adjust size here

sns.heatmap(
    corr_matrix_rounded,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    annot_kws={"size": 8},   # smaller numbers so they fit
    linewidths=0.5
)

plt.title("Correlation: Engineered Numeric Features", fontsize=16)
plt.xticks(rotation=60, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



The correlation heatmap is basically a redundancy detector (which features to drop).
High correlations indicate features we should drop (especially for linear models) like below:
- Volatility_5D ↔ Volatility_10D
- Rolling_Avg_Close_50D ↔ Rolling_Avg_Close_200D
- Event_Vol_5D ↔ Event_Vol_10D

But low correlation does not imply usefulness — feature importance from the model determines that.
So correlation is mainly for removing duplicates, not selecting predictors.

In [ ]:
terror_dates = df_events[df_events['Event_Type_Mapped'] == 'terror']['Event_Date']
window = 30  # 30-day event window
curves = []

for event_date in terror_dates:
    # Make sure date exists
    if event_date not in df_data_index['Date'].values:
        continue

    # Slice the next 30 days
    slice_df = df_data_index[df_data_index['Date'] >= event_date].head(window + 1)

    if len(slice_df) < window + 1:
        continue  # skip if not enough future data

    # Compute returns relative to event date close
    base_price = slice_df['Close'].iloc[0]
    cum_return = (slice_df['Close'] / base_price) - 1

    curves.append(cum_return.reset_index(drop=True))

reaction_df = pd.DataFrame(curves)
avg_reaction = reaction_df.mean()

plt.figure(figsize=(10,6))
plt.plot(avg_reaction.index, avg_reaction.values, linewidth=3)
plt.axhline(0, color='black', linewidth=1)
plt.title("Average 30-Day Reaction to Terror Events")
plt.xlabel("Days After Event")
plt.ylabel("Cumulative Return")
plt.grid(True, alpha=0.3)
plt.show()


Do a crosstab of Region and Currency categorical features

In [ ]:
pd.crosstab(df_data_index["Region"], df_data_index["Currency"])

Create Clusters

In [ ]:
# engineered features to aggregate
eng_features = [
    'Daily_Return', 'Volatility_5D', 'Volatility_10D',
    'Rolling_Avg_Close_50D', 'Rolling_Avg_Close_200D',
    'Volume_Change', 'Range', 'Intraday_Volatility'
]

# Replace infinities with NaN since this will interfere with clustring
df_data_index.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop all rows with NaNs???
df_num = df_data_index[['Index'] + eng_features].dropna()


# filter only numeric features (dropping NaNs from rolling windows)
df_num = df_data_index[['Index'] + eng_features].dropna()

# compute mean values per Index
df_index_features = df_num.groupby("Index")[eng_features].mean().reset_index()

df_index_features


In [ ]:
# ensure nans fixed
numeric_only = df_index_features.select_dtypes(include=[np.number])

print(np.isinf(numeric_only).sum())
print(df_index_features.isna().sum())


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_index_features[eng_features])


In [ ]:
from sklearn.cluster import KMeans
# Came back after the elbow curve and changed clusters to 6
#kmeans = KMeans(n_clusters=3, random_state=42)
kmeans = KMeans(n_clusters=6, random_state=42)
df_index_features['Cluster'] = kmeans.fit_predict(X_scaled)

df_index_features


In [ ]:
cluster_profiles = df_index_features.groupby("Cluster")[eng_features].mean()
cluster_profiles

In [ ]:
from sklearn.cluster import KMeans
inertia = []

for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)


plt.plot(range(2,10), inertia, marker='o')
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.show()


We see that after 5-6 there's stops being a significant change in inertia (How far points are from their assigned cluster centers.) So we'll select k=6.

Next we'll visualize the boxplot of the clusters.

In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(cluster_profiles, annot=True, cmap="coolwarm")
plt.title("Cluster Feature Profiles")
plt.show()


In [ ]:
cluster_profiles

Cluster 0 contains the largest, highest-priced indexes with moderate volatility.
Cluster 3 represents low-cost, low-volatility indexes.
Cluster 4 is uniquely high-volatility, suggesting a more speculative environment.

## Step 3:  Identify 1-3 research questions and perform analysis

Now that you have a better understanding of the data, you will want to form a research question which is interesting to you. The research question should be broad enough to be of interest to a reader but narrow enough that the question can be answered with the data.  Some examples:

* __Too Narrow:__  What is the GDP of the U.S. for 2011?  This is just asking for a fact or a single data point.  

* __Too Broad:__  What is the primary reason for global poverty?  This could be a Ph.D. thesis and would still be way too broad.  What data will you use to answer this question?  Even if a single dataset offered an answer, would it be defendable given the variety of datasets out there?

* __Good:__  Can you use simple sentiment analysis on comments about movies in a movie database to predict its box office earnings?  If you have, or can obtain, data on a variety of movies and you have their box office earnings, this is a question which you can potentially answer well.

__Remember__, this course is for learning Python. You will not be graded on the complexity, accuracy or performance of your analytical methods. However, you are expected to use a Python library, e.g., _scikitlearn_, successfully to generate results and explain why you picked the methods you used.



In [ ]:
#OVERVIEW YOUR QUESTION AND PERFORM YOUR ANALYSIS IN THIS SECTION

## Step 4:  Present your findings

In this step, you can begin to report your findings.  What did you learn from the data and how do your findings help answer your research question?  Use _matplotlib_ visualizations to present these findings.


__Remember:__ Rarely will a single data analysis conclusively answer a research question.  Here, you need to identify possible limitations.  For example, are your results limited to a certain area, city, or country?  Are you making assumptions about the data which may, or may not, be valid (e.g., that students in one term are equally qualified as students in another)?  Document these limitations in a few paragraphs.


In [ ]:
#EXPAND THIS SECTION TO PRESENT YOUR FINDINGS